Going to try with matplot lib

I used chatgpt to help me generate some of this initial code

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors
import plotly.express as px


Read in dataframes, and then make a list of dataframes so I can run through them all

In [2]:
#election years
df_2005 = pd.read_csv('DC-migration-2005.csv')
df_2009 = pd.read_csv('DC-migration-2009.csv')
df_2013 = pd.read_csv('DC-migration-2013.csv')
df_2017 = pd.read_csv('DC-migration-2017.csv')
df_2021 = pd.read_csv('DC-migration-2021.csv')


#pre-election years for comparison

df_2015 = pd.read_csv('DC-migration-2015.csv')
df_2016 = pd.read_csv('DC-migration-2016.csv')
df_2018 = pd.read_csv('DC-migration-2018.csv')
df_2019 = pd.read_csv('DC-migration-2019.csv')



Drop the initial columns that were carried over from the indexes and messing up this data.

In [3]:
df_2005 = df_2005.drop(columns=["Unnamed: 0"])
df_2009 = df_2009.drop(columns=["Unnamed: 0"])
df_2013 = df_2013.drop(columns=["Unnamed: 0"])
df_2017 = df_2017.drop(columns=["Unnamed: 0"])
df_2021 = df_2021.drop(columns=["Unnamed: 0"])

df_2015 = df_2015.drop(columns=["Unnamed: 0"])
df_2016 = df_2016.drop(columns=["Unnamed: 0"])
df_2018 = df_2018.drop(columns=["Unnamed: 0"])
df_2019 = df_2019.drop(columns=["Unnamed: 0"])


In [4]:
df_2005.tail()

df_2015.tail()

,state,pop_migrated,MOE +/-,state_population,migrated_adjusted,moe_adjusted
43,Vermont,117,143,643077,0.000182,0.000222
44,Washington,134,164,7705281,0.000017,0.000021
45,West Virginia,478,432,1793716,0.000266,0.000241
46,Wisconsin,514,421,5893718,0.000087,0.000071
47,Wyoming,0,200,576851,0.000000,0.000347


Add in a state year column so in order to be able to loop through the dataframes


In [5]:
df_2005['year'] = 2005
df_2009['year'] = 2009
df_2013['year'] = 2013
df_2017['year'] = 2017
df_2021['year'] = 2021

df_2015['year'] = 2015
df_2016['year'] = 2016
df_2018['year'] = 2018
df_2019['year'] = 2019

Forming a new dataframe with 2015, 2016, 2017 data

In [6]:
df_2015_16_17_compare = pd.DataFrame()

df_2015_16_17_compare['state'] = df_2017['state'].copy()
df_2015_16_17_compare['pop_migrated_2015'] = df_2015['pop_migrated'].copy()
df_2015_16_17_compare['pop_migrated_2016'] = df_2016['pop_migrated'].copy()
df_2015_16_17_compare['pop_migrated_2017'] = df_2017['pop_migrated'].copy()


Make a new column averaging the non election years -- i.e. 2015 and 2016 (since census is early in year before election)

In [7]:
df_2015_16_17_compare['15_16_avg'] = (df_2015_16_17_compare['pop_migrated_2015'] + df_2015_16_17_compare['pop_migrated_2016']) / 2

In [8]:
df_2015_16_17_compare.head(30)

,state,pop_migrated_2015,pop_migrated_2016,pop_migrated_2017,15_16_avg
0,Alabama,359,140,494,249.5
1,Alaska,235,77,0,156.0
2,Arizona,367,744,549,555.5
3,Arkansas,203,0,105,101.5
4,California,4119,3908,3763,4013.5
5,Colorado,433,531,1109,482.0
6,Connecticut,1292,1225,1149,1258.5
7,Delaware,252,278,375,265.0
8,Florida,2731,2717,1182,2724.0
9,Georgia,1013,1105,1507,1059.0


Make a new column with the percent change between 2016 and 2017 years

In [9]:
df_2015_16_17_compare['pct_change_pre_post_election'] = (
    df_2015_16_17_compare['pop_migrated_2017'] - df_2015_16_17_compare['15_16_avg']
) / df_2015_16_17_compare["15_16_avg"] * 100

df_2015_16_17_compare.head()

,state,pop_migrated_2015,pop_migrated_2016,pop_migrated_2017,15_16_avg,pct_change_pre_post_election
0,Alabama,359,140,494,249.5,97.995992
1,Alaska,235,77,0,156.0,-100.000000
2,Arizona,367,744,549,555.5,-1.170117
3,Arkansas,203,0,105,101.5,3.448276
4,California,4119,3908,3763,4013.5,-6.241435


Add back in the party info

In [10]:
df_2015_16_17_compare['winning_party'] = df_2017['winning_party'] 
df_2015_16_17_compare['winner_percent_of_votes'] = df_2017['winner_percent_of_votes'] 

df_2015_16_17_compare.head()

,state,pop_migrated_2015,pop_migrated_2016,pop_migrated_2017,15_16_avg,pct_change_pre_post_election,winning_party,winner_percent_of_votes
0,Alabama,359,140,494,249.5,97.995992,REPUBLICAN,0.620831
1,Alaska,235,77,0,156.0,-100.000000,REPUBLICAN,0.512815
2,Arizona,367,744,549,555.5,-1.170117,REPUBLICAN,0.486716
3,Arkansas,203,0,105,101.5,3.448276,REPUBLICAN,0.605741
4,California,4119,3908,3763,4013.5,-6.241435,DEMOCRAT,0.617264


Add a total column, so that I can see tota numbers migrating into D.C. in an election year and not in an election year. 

In [11]:
df_2015_16_17_compare.loc["Total"] = df_2015_16_17_compare.sum(numeric_only=True)

df_2015_16_17_compare.head()

,state,pop_migrated_2015,pop_migrated_2016,pop_migrated_2017,15_16_avg,pct_change_pre_post_election,winning_party,winner_percent_of_votes
0,Alabama,359.0,140.0,494.0,249.5,97.995992,REPUBLICAN,0.620831
1,Alaska,235.0,77.0,0.0,156.0,-100.000000,REPUBLICAN,0.512815
2,Arizona,367.0,744.0,549.0,555.5,-1.170117,REPUBLICAN,0.486716
3,Arkansas,203.0,0.0,105.0,101.5,3.448276,REPUBLICAN,0.605741
4,California,4119.0,3908.0,3763.0,4013.5,-6.241435,DEMOCRAT,0.617264


In [12]:
df_2015_16_17_compare.sort_values('pct_change_pre_post_election', ascending=False)

,state,pop_migrated_2015,pop_migrated_2016,pop_migrated_2017,15_16_avg,pct_change_pre_post_election,winning_party,winner_percent_of_votes
Total,NaN,37151.0,35529.0,37887.0,36340.0,7259.628037,NaN,26.348198
11,Idaho,5.0,0.0,161.0,2.5,6340.000000,REPUBLICAN,0.592614
29,New Mexico,40.0,1.0,206.0,20.5,904.878049,DEMOCRAT,0.482556
43,Vermont,117.0,0.0,447.0,58.5,664.102564,DEMOCRAT,0.557227
5,Colorado,433.0,531.0,1109.0,482.0,130.082988,DEMOCRAT,0.481570
0,Alabama,359.0,140.0,494.0,249.5,97.995992,REPUBLICAN,0.620831
42,Utah,177.0,249.0,384.0,213.0,80.281690,REPUBLICAN,0.455408
12,Illinois,785.0,2418.0,2726.0,1601.5,70.215423,DEMOCRAT,0.558254
41,Texas,1314.0,2213.0,2858.0,1763.5,62.064077,REPUBLICAN,0.522347
40,Tennessee,873.0,452.0,1057.0,662.5,59.547170,REPUBLICAN,0.607220


Forming a new dataframe with 2018 and 2019, and 2021 data (there is no 2018 instead of 2020 because no data in 2020 because of covid)

In [13]:
df_2018_19_21_compare = pd.DataFrame()

df_2018_19_21_compare['state'] = df_2019['state'].copy()
df_2018_19_21_compare['pop_migrated_2018'] = df_2018['pop_migrated'].copy()
df_2018_19_21_compare['pop_migrated_2019'] = df_2019['pop_migrated'].copy()
df_2018_19_21_compare['pop_migrated_2021'] = df_2021['pop_migrated'].copy()


Make a column averaging 2018 and 2019

In [14]:
df_2018_19_21_compare['18-19_avg'] = (df_2018_19_21_compare['pop_migrated_2018'] + df_2018_19_21_compare['pop_migrated_2019']) / 2

Add back in the party info

In [15]:
df_2018_19_21_compare['winning_party'] = df_2021['winning_party'] 
df_2018_19_21_compare['winner_percent_of_votes'] = df_2021['winner_percent_of_votes'] 

df_2018_19_21_compare.head()

,state,pop_migrated_2018,pop_migrated_2019,pop_migrated_2021,18-19_avg,winning_party,winner_percent_of_votes
0,Alabama,174,462,342,318.0,REPUBLICAN,0.620316
1,Alaska,0,338,196,169.0,REPUBLICAN,0.528331
2,Arizona,204,291,674,247.5,DEMOCRAT,0.493647
3,Arkansas,0,0,37,0.0,REPUBLICAN,0.623957
4,California,3803,3437,3945,3620.0,DEMOCRAT,0.634839


Percent change 2019 vs 2021 of migrants to DC from each state

In [16]:
df_2018_19_21_compare['pct_change_pre_post_election'] = (
    df_2018_19_21_compare['pop_migrated_2021'] - df_2018_19_21_compare['18-19_avg']
) / df_2018_19_21_compare["18-19_avg"] * 100

df_2018_19_21_compare.head()

,state,pop_migrated_2018,pop_migrated_2019,pop_migrated_2021,18-19_avg,winning_party,winner_percent_of_votes,pct_change_pre_post_election
0,Alabama,174,462,342,318.0,REPUBLICAN,0.620316,7.547170
1,Alaska,0,338,196,169.0,REPUBLICAN,0.528331,15.976331
2,Arizona,204,291,674,247.5,DEMOCRAT,0.493647,172.323232
3,Arkansas,0,0,37,0.0,REPUBLICAN,0.623957,inf
4,California,3803,3437,3945,3620.0,DEMOCRAT,0.634839,8.977901


Add in a total column so that I can see tota numbers migrating into D.C. in an election year and not in an election year. 

In [17]:
df_2018_19_21_compare.loc["Total"] = df_2018_19_21_compare.sum(numeric_only=True)

df_2018_19_21_compare.head()

,state,pop_migrated_2018,pop_migrated_2019,pop_migrated_2021,18-19_avg,winning_party,winner_percent_of_votes,pct_change_pre_post_election
0,Alabama,174.0,462.0,342.0,318.0,REPUBLICAN,0.620316,7.547170
1,Alaska,0.0,338.0,196.0,169.0,REPUBLICAN,0.528331,15.976331
2,Arizona,204.0,291.0,674.0,247.5,DEMOCRAT,0.493647,172.323232
3,Arkansas,0.0,0.0,37.0,0.0,REPUBLICAN,0.623957,inf
4,California,3803.0,3437.0,3945.0,3620.0,DEMOCRAT,0.634839,8.977901


In [18]:
df_2018_19_21_compare.sort_values('pct_change_pre_post_election', ascending=False)

,state,pop_migrated_2018,pop_migrated_2019,pop_migrated_2021,18-19_avg,winning_party,winner_percent_of_votes,pct_change_pre_post_election
Total,NaN,27047.0,34552.0,40989.0,30799.5,NaN,27.582650,inf
3,Arkansas,0.0,0.0,37.0,0.0,REPUBLICAN,0.623957,inf
7,Delaware,182.0,0.0,3560.0,91.0,DEMOCRAT,0.587430,3812.087912
15,Kansas,0.0,41.0,503.0,20.5,REPUBLICAN,0.562125,2353.658537
26,Nevada,36.0,0.0,431.0,18.0,DEMOCRAT,0.500568,2294.444444
11,Idaho,0.0,28.0,93.0,14.0,REPUBLICAN,0.638376,564.285714
34,Oklahoma,0.0,185.0,374.0,92.5,REPUBLICAN,0.653733,304.324324
18,Maine,61.0,448.0,1028.0,254.5,DEMOCRAT,0.525256,303.929273
21,Minnesota,143.0,278.0,726.0,210.5,DEMOCRAT,0.523951,244.893112
35,Oregon,80.0,172.0,426.0,126.0,DEMOCRAT,0.564533,238.095238


In [19]:
df_2018_19_21_compare.head()

,state,pop_migrated_2018,pop_migrated_2019,pop_migrated_2021,18-19_avg,winning_party,winner_percent_of_votes,pct_change_pre_post_election
0,Alabama,174.0,462.0,342.0,318.0,REPUBLICAN,0.620316,7.547170
1,Alaska,0.0,338.0,196.0,169.0,REPUBLICAN,0.528331,15.976331
2,Arizona,204.0,291.0,674.0,247.5,DEMOCRAT,0.493647,172.323232
3,Arkansas,0.0,0.0,37.0,0.0,REPUBLICAN,0.623957,inf
4,California,3803.0,3437.0,3945.0,3620.0,DEMOCRAT,0.634839,8.977901


New method to see anomolies


In [20]:
df_2015_16_17_compare['avg_all years'] = (df_2005['pop_migrated'] + df_2009['pop_migrated'] + df_2013['pop_migrated'] + df_2021['pop_migrated'] + df_2015['pop_migrated'] + df_2016['pop_migrated'] + df_2018['pop_migrated'] + df_2019['pop_migrated']) / 8


df_2015_16_17_compare['anomoly'] = df_2015_16_17_compare['pop_migrated_2017'] / df_2015_16_17_compare['avg_all years']

In [21]:
df_2018_19_21_compare['avg_all years'] = (df_2005['pop_migrated'] + df_2009['pop_migrated'] + df_2013['pop_migrated'] + df_2017['pop_migrated'] + df_2015['pop_migrated'] + df_2016['pop_migrated'] + df_2018['pop_migrated'] + df_2019['pop_migrated']) / 8


df_2018_19_21_compare['anomoly'] = df_2018_19_21_compare['pop_migrated_2021'] / df_2018_19_21_compare['avg_all years']

Look at anomolies

In [22]:
df_2018_19_21_compare.sort_values('anomoly', ascending=False)

,state,pop_migrated_2018,pop_migrated_2019,pop_migrated_2021,18-19_avg,winning_party,winner_percent_of_votes,pct_change_pre_post_election,avg_all years,anomoly
15,Kansas,0.0,41.0,503.0,20.5,REPUBLICAN,0.562125,2353.658537,21.125,23.810651
7,Delaware,182.0,0.0,3560.0,91.0,DEMOCRAT,0.587430,3812.087912,293.875,12.113994
18,Maine,61.0,448.0,1028.0,254.5,DEMOCRAT,0.525256,303.929273,196.375,5.234882
44,Washington,387.0,852.0,1817.0,619.5,DEMOCRAT,0.579703,193.301049,348.625,5.211904
26,Nevada,36.0,0.0,431.0,18.0,DEMOCRAT,0.500568,2294.444444,91.875,4.691156
11,Idaho,0.0,28.0,93.0,14.0,REPUBLICAN,0.638376,564.285714,24.250,3.835052
34,Oklahoma,0.0,185.0,374.0,92.5,REPUBLICAN,0.653733,304.324324,99.500,3.758794
47,Wyoming,68.0,0.0,56.0,34.0,REPUBLICAN,0.694998,64.705882,15.625,3.584000
27,New Hampshire,130.0,49.0,266.0,89.5,DEMOCRAT,0.527078,197.206704,91.000,2.923077
35,Oregon,80.0,172.0,426.0,126.0,DEMOCRAT,0.564533,238.095238,153.250,2.779772


In [23]:
df_2015_16_17_compare.sort_values('anomoly', ascending=False)


,state,pop_migrated_2015,pop_migrated_2016,pop_migrated_2017,15_16_avg,pct_change_pre_post_election,winning_party,winner_percent_of_votes,avg_all years,anomoly
11,Idaho,5.0,0.0,161.0,2.5,6340.000000,REPUBLICAN,0.592614,15.750,10.222222
43,Vermont,117.0,0.0,447.0,58.5,664.102564,DEMOCRAT,0.557227,71.500,6.251748
5,Colorado,433.0,531.0,1109.0,482.0,130.082988,DEMOCRAT,0.481570,442.750,2.504800
3,Arkansas,203.0,0.0,105.0,101.5,3.448276,REPUBLICAN,0.605741,43.250,2.427746
41,Texas,1314.0,2213.0,2858.0,1763.5,62.064077,REPUBLICAN,0.522347,1413.500,2.021931
13,Indiana,748.0,957.0,710.0,852.5,-16.715543,REPUBLICAN,0.569400,355.500,1.997187
16,Kentucky,275.0,233.0,285.0,254.0,12.204724,REPUBLICAN,0.625196,142.875,1.994751
12,Illinois,785.0,2418.0,2726.0,1601.5,70.215423,DEMOCRAT,0.558254,1376.000,1.981105
0,Alabama,359.0,140.0,494.0,249.5,97.995992,REPUBLICAN,0.620831,249.500,1.979960
42,Utah,177.0,249.0,384.0,213.0,80.281690,REPUBLICAN,0.455408,207.875,1.847264


Export

In [24]:
df_2015_16_17_compare.to_csv('df_2015_16_17_compare.csv')
df_2018_19_21_compare.to_csv('df_2015_16_17_compare.csv')

Make some maps of the anomolies for 2017 and 2021

In [25]:
import numpy as np
import matplotlib.colors as mcolors



cleaning for mapping

In [26]:
df_2017_formap = df_2015_16_17_compare.sort_values('anomoly', ascending=False).head(15)
df_2021_formap = df_2018_19_21_compare.sort_values('anomoly', ascending=False).head(15)


In [27]:
us_state_abbrev = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC"
}

for i in range(len(df_2017_formap)):
    df_2017_formap["state_code"] = df_2017_formap["state"].map(us_state_abbrev)
for i in range(len(df_2021_formap)):
    df_2021_formap["state_code"] = df_2021_formap["state"].map(us_state_abbrev)


Making new column for shading on map

In [32]:
for i in range(len(df_2017_formap)):
    df_2017_formap["percent_won_and_party"] = np.where(
        df_2017_formap["winning_party"] == "DEMOCRAT",
        df_2017_formap["winner_percent_of_votes"],    # keep as-is (positive)
        df_2017_formap["winner_percent_of_votes"] * -1    # flip sign to negative
    )

for i in range(len(df_2021_formap)):
    df_2021_formap["percent_won_and_party"] = np.where(
        df_2021_formap["winning_party"] == "DEMOCRAT",
        df_2021_formap["winner_percent_of_votes"],    # keep as-is (positive)
        df_2021_formap["winner_percent_of_votes"] * -1   # flip sign to negative
    )

In [33]:
df_2021_formap.head()

,state,pop_migrated_2018,pop_migrated_2019,pop_migrated_2021,18-19_avg,winning_party,winner_percent_of_votes,pct_change_pre_post_election,avg_all years,anomoly,state_code,percent_won_and_party
15,Kansas,0.0,41.0,503.0,20.5,REPUBLICAN,0.562125,2353.658537,21.125,23.810651,KS,-0.562125
7,Delaware,182.0,0.0,3560.0,91.0,DEMOCRAT,0.587430,3812.087912,293.875,12.113994,DE,0.587430
18,Maine,61.0,448.0,1028.0,254.5,DEMOCRAT,0.525256,303.929273,196.375,5.234882,ME,0.525256
44,Washington,387.0,852.0,1817.0,619.5,DEMOCRAT,0.579703,193.301049,348.625,5.211904,WA,0.579703
26,Nevada,36.0,0.0,431.0,18.0,DEMOCRAT,0.500568,2294.444444,91.875,4.691156,NV,0.500568


In [ ]:
fig_2021_anomoly = px.choropleth(
    df_2021_formap,
    locations="state_code",
    locationmode="USA-states",
    color="percent_won_and_party",
    scope="usa",
    color_continuous_scale=["#AA0000", '#C88FE4',"#0645B4"],
    range_color=[-0.8, 0.8]
)

fig_2021_anomoly.show()


In [ ]:
fig_2017_anomoly = px.choropleth(
    df_2017_formap,
    locations="state_code",
    locationmode="USA-states",
    color="percent_won_and_party",
    scope="usa",
    color_continuous_scale=["#AA0000", '#C88FE4',"#0645B4"],
    range_color=[-0.8, 0.8]
)

fig_2017_anomoly.show()

Save new maps

In [ ]:
fig_2017_anomoly.write_image("2009_map.svg", engine="kaleido")
fig_2021_anomoly.write_image("2021_map.svg", engine="kaleido")

print("Map successfully exported as 'choropleth_map.subplots.svg'!")